# Notebook 4A: Introduction to Supervised Learning with CalEnviroScreen

*Authored by Dr. Noelle Anderson in 2026*

## Introduction

In this notebook, we are shifting from **unsupervised learning** to **supervised learning**!

Earlier in the course, you used unsupervised methods like PCA, k-means, and hierarchical clustering to explore patterns in data without choosing an answer column to predict. That is useful when we want to discover data structure, group similar samples, or reduce a large dataset down to a smaller set of summary patterns.

Now we are moving into a different kind of machine learning question: **can we predict an outcome using other columns in the dataset?**

To practice that shift to supervised learning, we will work with a new dataset, the **CalEnviroScreen 5.0** dataset, a California environmental health dataset. Each row describes one small geographic area, and the columns include environmental exposures, social conditions, demographic patterns, and health-related measures.

This notebook is mostly about **setting up** a supervised learning problem, we will not fit a model yet. By the end, you should be able to:

- explain the difference between **supervised** and **unsupervised** learning
- describe the difference between predicting a **number** and predicting a **category**
- identify a **target**, which is the column we want to predict
- identify **features**, which are the columns a model can use to make predictions
- recognize columns that are useful for context but not good first-choice model inputs
- explain why some columns can cause **data leakage**
- create `X` and `y`, the standard sklearn names for features and target
- split rows into training and test sets for later modeling

Some parts may feel new, but the workflow should also feel familiar: load data, inspect it, make careful choices, and prepare the data in a clear order.

## Supervised vs. unsupervised learning

In the first half of SCIP, you practiced **unsupervised learning**. Now we are starting **supervised learning**. Here is the simplest way to think about the difference:

- **Unsupervised learning** starts with data but **no answer column**. We look for patterns, clusters, or lower-dimensional structure.
- **Supervised learning** starts with data **and** a known answer column. The goal is to learn a relationship between the other columns and that answer so we can make predictions.

Some supervised learning problems predict a **number**. These are called **regression** problems. Other supervised learning problems predict a **category**. These are called **classification** problems.

A few examples:

- In **biology**, you might use unsupervised learning to group cells with similar gene-expression profiles, or supervised learning to predict whether a sample has a high or low biomarker value.
- In **biochemistry**, you might use unsupervised learning to explore patterns across compounds, or supervised learning to predict enzyme activity from experimental measurements.
- In **environmental science**, you might use unsupervised learning to look for site-level patterns, or supervised learning to predict a measured environmental burden.
- In **public health**, you might explore whether communities group together by burden, or you might predict a tract-level health outcome using environmental and social conditions.

So the shift is not just technical, it is also about the kind of question we are asking.

## Notebook 4A roadmap

In this notebook, we will follow this workflow:

1. **Load the CalEnviroScreen dataset**
2. **Explore the dataset structure and column types**
3. **Identify context columns and possible health outcome columns**
4. **Use `Asthma` as a guided example target**
5. **Learn the vocabulary of targets, labels, and features**
6. **Identify columns that should not be used as ordinary features in this first workflow**
7. **Introduce data leakage**
8. **Remove rows with missing target values**
9. **Build `X` features and `y` target data for sklearn**
10. **Split the rows into training and test sets**
11. **Preview why supervised preprocessing happens after the split**


*Note: This week's notebook may feel longer and more challenging. This is intentional because 1) you are ready for it by this time in the course, and 2) we are both getting to know a new dataset and getting to know a whole different way of doing ML (supervised learning), so expect a bit more reading and code.*

## Step 0: Import needed packages

In [ ]:
# These are packages you have already seen in earlier notebooks.
import pandas as pd
import matplotlib.pyplot as plt

# This function helps us split rows into training and test sets.
from sklearn.model_selection import train_test_split

# Introducing our new dataset: CalEnviroScreen

In the first half of the course, you worked with the METABRIC dataset, where each row represented one patient. In this part of the course, we are switching to a subset of **CalEnviroScreen 5.0**, created by the California Office of Environmental Health Hazard Assessment.

In this dataset, each row represents a **census tract** rather than an individual person. A census tract is a small geographic area defined by the U.S. Census Bureau, so this dataset lets us study health and environmental conditions at the community level.

CalEnviroScreen combines data from multiple sources to describe conditions across census tracts in California. In the version we will use, each row includes measures related to pollution, public health, and community conditions. This means we are no longer asking per-patient biomedical questions. Instead, we are asking community-level questions about how environmental burdens and social conditions are distributed across places.

This shift matters because it connects directly to **environmental justice**, which is the idea that all communities should have equal protection from environmental harm and equal access to healthy environments. In practice, environmental justice asks us to pay attention to the fact that pollution, unsafe water, housing burden, disinvestment, and other structural harms are not distributed evenly. These patterns are shaped by histories of policy, land use, racism, class inequality, labor conditions, and unequal political power.

Because these topics affect real communities, including communities that students may belong to or care about, we will be careful about interpretation. We will use this dataset to practice machine learning, but we will not treat model inputs as simple causes or treat community-level data as if it explains individual people.

[Learn more about environmental justice in CalEnviroScreen in this short video here.](https://www.youtube.com/watch?v=J8NpXb2qfK8&list=PLELCwL6ywxq2Z_XVXZYIIPxLT0UDglK17&index=6)

CalEnviroScreen is also important because it is used in California policy. It was designed to help identify communities facing high pollution burden and population vulnerability, so the data has real-world meaning beyond this class.

## Step 1: Load the CalEnviroScreen data from Google Drive

We will load the data from a public Google Drive link, similar to how you loaded data earlier in the course.

Like `patient_id` in the METABRIC notebooks, `Census Tract` is a unique identifier in this dataset, so it makes sense to use it as the dataframe index. The columns are described in more detail below.

In [ ]:
# Public Google Drive file ID provided by staff.
file_id = "1X4-6X3VKhR2jRHppI3XuVV79nhHyg8Xb"

# Direct download link for pandas.
url = f"https://drive.google.com/uc?export=download&id={file_id}"

# Load the dataset and use Census Tract as the row index.
ces = pd.read_csv(url, index_col="Census Tract")

display(ces.head())
print("\nCalEnviroScreen dimensions:")
display(ces.shape)

Before we go further, pause and notice the size of the dataset.

We have many census tracts and many different kinds of columns mixed together in one table. That is common in real public health and environmental datasets, and it means we need to be thoughtful about what role each column should play in our analysis.

## Step 2: Explore the dataset structure

A dataset can look manageable at first glance and still contain several different kinds of columns. Let's inspect the columns before we do more.

### <font color=0D9488>**Question 1:**</font> Print the dataset info summary so you can check the full column names, non-null counts, and data types.

In [ ]:
# Your solution here!

## Understanding the columns in the CalEnviroScreen dataset

This notebook is largely about learning how to turn a real dataset into a supervised learning problem, so it is worth slowing down here. As you look through the columns, keep these questions in mind:

- Which columns look like possible **health outcomes** we might want to predict?
- Which columns look like possible **features** that a model could use for prediction?
- Which columns look more like **identifiers or location information**, which may be useful for context but not for ordinary prediction?
- Which columns look like **scores, percentiles, or summary variables** that may already be built from other information in the dataset and therefore need extra caution?

Because `Census Tract` is our dataframe index, each row represents one **census tract** rather than one individual person like in METABRIC. A census tract is a small geographic area defined by the U.S. Census Bureau for organizing population data. It usually contains a few thousand residents and is designed to represent a relatively stable local area, which makes it useful for comparing community conditions across places.

[You can look up your own exact census tract using your address here. Scroll down to "Census Tracts" and look for the "GEOID". Note that in our dataset, it will not have the 0 at the beginning.](https://geocoding.geo.census.gov/geocoder/geographies/address?form)

### Context columns
Some fields help identify or locate each tract.

- **California County**, **ZIP**, and **Approximate Location** tell us where the tract is.
- **Census Tract** is the unique label for that geographic area, and here we are using it as the dataframe index rather than as an ordinary feature column.
- **Total Population** is different. It is an actual tract-level measurement, so we will keep it in the dataset as one possible feature.

### Environmental and community columns
Many columns describe conditions in the tract.

Environmental burden columns include:
- **Ozone** and **PM2.5**: air pollution
- **Diesel PM**: diesel particle pollution nearby
- **Drinking Water**: an index related to drinking water contaminants
- **Pesticides**: pesticide use in production agriculture
- **Traffic**: traffic density
- **Cleanup Sites**: burden from nearby hazardous waste cleanup and Superfund sites

Community condition columns include:
- **Education**: percent of adults over 25 with less than a high school education
- **Housing Burden**: percent of households that are low-income and heavily burdened by housing costs
- **Linguistic Isolation**: percent of households where no one over age 14 speaks English well
- **Poverty**: percent of people living below twice the federal poverty level
- **Unemployment**: percent of people age 16 and older who are unemployed and in the workforce

### Health outcome columns
These columns measure health-related outcomes:
- **Asthma**
- **Cardiovascular Disease**
- **Diabetes**
- **Low-Birth-Weight**

### Demographic columns
These columns describe age structure and race or ethnicity percentages at the tract level. These variables can matter for environmental justice because racialized and age-related patterns are often connected to unequal exposure, housing, labor conditions, health care access, and policy histories. At the same time, we must interpret them carefully since race is not a biological explanation for asthma burden or environmental burden. Race-related variables can reflect structural conditions and unequal treatment, not inherent biological differences.

### Scores, percentiles, and summary columns
Some columns are scores, percentiles, or summary measures. These can be useful for understanding the dataset, but they need extra caution before modeling because they may be built from other columns or may summarize the outcome too directly.

---

The example below shows one way to use this dataset to ask a place-based question. This is not the main modeling workflow yet, it is just a short example to help you see that the rows and columns describe real places we know.

In [ ]:
# Group the data by county and calculate the average pesticide burden
# across all census tracts in each county.
county_pesticides = (
    ces.groupby("California County")["Pesticides"] # groupby is very powerful!
    .mean()
    .sort_values(ascending=False)
    .head(15)
)

# Create a figure of a reasonable size for a horizontal bar chart.
plt.figure(figsize=(8, 5))

# Plot the top 15 counties. Sorting again here puts the bars in
# ascending order so the largest value appears at the top of the chart.
county_pesticides.sort_values().plot(kind="barh")

# Add a clear title and axis labels so students know what is being shown.
plt.title("Top 15 Counties by Average Pesticide Burden")
plt.xlabel("Average Pesticides")
plt.ylabel("California County")

# Display the finished plot.
plt.show()

**What do you notice in this plot? Does the pattern seem surprising, or expected based on your own experience, or hard to interpret from this plot alone?**

Data like this can bring up realities that are not abstract, environmental burdens affect real places and real communities, including communities that we may belong to or care about personally. This dataset can hit very close to home and is both powerful and potentially heavy to work with. Because of that, we want to approach this dataset and this course with care.

Pollution, disinvestment, unequal burden, and health harm do not happen by accident or affect all communities evenly. They are shaped by policy, power, racism, class inequality, land use, labor conditions, and other structural forces. At the same time, one plot is not enough to explain why a pattern exists or what should be done about it.

As we move into modeling, we will keep two ideas in mind:

- The data can help us study patterns across communities.
- A model pattern is not the same thing as a complete explanation of why those patterns exist.

You are welcome to explore the data further in ways we do not show in this notebook and to ask your own questions, especially questions that connect the dataset back to place, policy, and lived experience. Data can be useful when we learn to work with it responsibly and with care.

## Step 3: Identify context columns

Before we choose an outcome to predict, we should separate columns that mainly **identify** a tract or **locate** it from columns that actually measure conditions we may want to model.

In this dataset, `California County`, `ZIP`, and `Approximate Location` mainly tell us **where** a tract is, and `Census Tract` tells us **which tract** a row refers to. These are useful for organizing the data, but they are not usually good first-choice features for a supervised learning model.

The main reason is that they act more like labels or location markers than like measurements of environmental burden, community conditions, or health. If we include them without thinking carefully, the model may learn to rely on place identity rather than on the tract-level conditions we actually want to study.

More generally, this is a good habit with any new dataset: separate columns that mainly identify, label, or provide context for observations from columns that actually measure characteristics that could be useful for your analysis or model.

### <font color=0D9488>**Question 2:**</font> Create a list called `context_cols` containing the columns `California County`, `ZIP`, and `Approximate Location`. Then display the first 5 rows of just those columns.

In [ ]:
# Your solution here!

## Step 4: Start thinking about possible prediction questions

Now that we have seen the kinds of columns in this dataset, we can start thinking about the kinds of supervised learning questions it could help us answer.

In supervised learning, we choose one column in our dataset as the thing we want to predict. Sometimes that means predicting a **number**, and sometimes it means predicting a **category**. We are not setting up a full model yet. For now, we are just getting used to the dataset and noticing which columns look like possible outcomes someone might want to predict later.

Focus on the health-related columns here. Which of them look like outcomes that could matter in a public health or environmental health context?

Let's make a list of a few to focus on.

In [ ]:
outcome_cols = [
    "Asthma",
    "Cardiovascular Disease",
    "Diabetes",
    "Low-Birth-Weight"
]

ces[outcome_cols].head()

## Step 5: Explore one outcome more closely

For the rest of this notebook, we will use **Asthma** as our main example outcome. In this dataset, `Asthma` represents the age-adjusted rate of emergency department visits for asthma. Age-adjusted means the measure has been adjusted so a tract does not look higher or lower just because its age distribution is different.

Asthma is a useful example here because it connects clearly to environmental health, public health, and the kinds of community conditions captured in CalEnviroScreen. It will also work well in the next notebook, when we begin with a supervised learning problem that predicts a numerical outcome.

### <font color=0D9488>**Question 3:**</font> Plot a histogram of `Asthma` with labeled axes and a title. Then print summary statistics for the same column using `describe()`.

In [ ]:
# Your solution here!

Notice here that the `Asthma` values vary across census tracts rather than staying the same everywhere. That matters because in supervised learning, we need an outcome that actually changes across observations.

We are also working with a community-level public health measure here, not an individual patient diagnosis. Each value summarizes a tract-level pattern rather than one person's health. That means we should interpret this column as part of a broader story about health and place.

Some conditions that can shape asthma patterns are reflected in this dataset, including air pollution, traffic, housing burden, poverty, age structure, and other community conditions. Those conditions are not distributed evenly, and in many places racist, classist, and other discriminatory policies like redlining have helped concentrate environmental harms and disinvestment in particular communities. [Read more about a recent study on childhood asthma and redlining here.](https://scienceblog.cincinnatichildrens.org/how-old-redlining-policies-still-affect-childhood-asthma-risk/)

So remember, when we analyze `Asthma` here, we are not trying to explain one person's asthma. We are studying how asthma burden varies across communities.

## Step 6: Learn supervised learning vocabulary

Now we can introduce some important supervised learning vocabulary. These are worth adding to your notes.

- In supervised machine learning, the column we want to predict is called the **target**.
- A target is also sometimes called a **label**, especially in classification problems.
- The columns we use to help make that prediction are called **features**.

A target can contain either **numbers** or **categories**.

- When we predict a **number**, the problem is called **regression**.
- When we predict a **category**, the problem is called **classification**.

In this notebook, we are preparing for a **regression** problem because `Asthma` is a numerical outcome. We are not fitting a model yet. Right now, the goal is to understand the data and organize it correctly for supervised learning.

## Step 7: Create a supervised-learning version of the dataset

Before we separate the data into a target and features, we need to think about column roles.

In this setup notebook, we are not choosing the final feature list for a fitted model yet. Instead, we are sorting columns into groups:

- the **target column**, which is the outcome we want to predict
- **context columns**, which mainly identify or locate each tract
- **other health outcome columns**, which we are leaving out in this first supervised setup
- **caution columns**, which are too closely tied to the `Asthma` outcome or are summary scores built from other columns
- **possible feature columns**, which are columns that could be considered as inputs in a supervised model

We are leaving out the other health outcome columns here, not because they are always invalid predictors, but because we want this first supervised workflow to stay focused on predicting one tract-level health outcome from environmental, demographic, and community-condition variables. Including other health outcomes like `Cardiovascular Disease`, `Diabetes`, or `Low-Birth-Weight` could make the model harder to interpret and blur the question we are trying to answer.

We also need extra caution with columns like:

- `Asthma Pctl`, because it is the percentile version of `Asthma`
- `Pop. Char. Score` and `Draft CES 5.0 Score`, because they are summary scores built from multiple parts of the dataset

## What is data leakage?

**Data leakage** happens when information that would not realistically be available at prediction time, or information that is too directly tied to the answer, slips into the model.

That is a problem because leakage can make a model seem much better than it really is. For example, if we are trying to predict `Asthma`, we should not include `Asthma Pctl` as a feature, because it is already calculated from the asthma column. The model would be getting a version of the answer.

At this stage, the key habit is simple: **slow down and ask what each column really means before using it.**

In [ ]:
# Provided code
# The target is the column we want to predict.
target_col = "Asthma"

# These columns need caution because they are too closely tied to Asthma
# or summarize multiple parts of the dataset.
caution_cols = ["Asthma Pctl", "Pop. Char. Score", "Draft CES 5.0 Score"]

# These are health outcomes that are not today's target.
other_outcome_cols = ["Cardiovascular Disease", "Diabetes", "Low-Birth-Weight"]

# These columns should not be treated as ordinary feature columns in this first setup.
not_feature_cols = [target_col] + context_cols + caution_cols + other_outcome_cols

# These are the remaining columns that could be considered possible features.
possible_feature_cols = [
    col for col in ces.columns
    if col not in not_feature_cols
]

# For this setup notebook, keep the target and the possible feature columns together.
ces_sup = ces[[target_col] + possible_feature_cols]

ces_sup.head()

At this point, `ces_sup` is our supervised-learning dataframe for this setup notebook. It contains the `Asthma` target and a broad set of possible feature columns.

This is not the final feature list for every fitted model. In Notebook 4B, we will make a narrower `model_features` list for the first regression model so that the model question stays focused and the first coefficient table is easier to interpret.

## Step 8: Check for missing data and drop missing target rows

Before we build `X` and `y`, we need to handle one important issue: some rows are missing the outcome value `Asthma`.

In supervised learning, we **cannot** keep rows where the target is missing, because the model would have no answer to learn from for those rows. So before moving forward, we will remove rows where `Asthma` is missing.

That is different from **feature missingness**. We may keep rows where some feature values are missing for now, because later we can decide how to handle those missing feature values during supervised preprocessing.

### <font color=0D9488>**Question 4:**</font> Check how many missing values are in the `Asthma` column of `ces_sup`.

In [ ]:
# Your solution here!

### <font color=0D9488>**Question 5:**</font> Create a new dataframe called `ces_sup_clean` by dropping rows where `Asthma` is missing. Then print the shapes of `ces_sup` and `ces_sup_clean`.

*Hint: use the `dropna()` command with the `subset` option.*

In [ ]:
# Your solution here!

Now that the target is complete, it is still a good habit to check how much missingness remains in the possible feature columns.

If a possible feature column had a very large amount of missing data, we might decide not to use it in a model. In this dataset, though, the possible feature columns have only a small amount of missingness, so we will keep them for this setup notebook and handle missing feature values later when needed.

In [ ]:
# Provided code
feature_missing_pct = ces_sup_clean[possible_feature_cols].isna().mean() * 100
feature_missing_pct = feature_missing_pct.round(2)

print("Percent missing in each possible feature column:")
print(feature_missing_pct)

A value like `0.71` means **0.71%** of rows are missing, not 71% missing, because the code multiplied the missing-value proportions by 100.

For this setup notebook, we will keep the possible feature columns. The important rule for today is that missing **target** values must be removed before supervised learning. Missing **feature** values can be handled later during preprocessing when needed.

---

**PAUSE:** There are two different kinds of data splitting happening next, and it is important not to mix them up:

- Step 9: When we create `X` and `y`, we are separating the **columns by role**. `X` will hold the feature columns, and `y` will hold the target column.
- Step 10: Then we will do a different kind of split with the **rows**. We will divide the rows into one group the model will learn from later and another group we will save for checking how well it works on rows it has not learned from.

We will do those as two separate steps below.

## Step 9: Build `X` for feature data and `y` for target data

Now that we have a cleaned supervised-learning dataframe, we are ready to separate the columns by role.

- `X` will hold the possible feature columns, which are the input columns that could be used for prediction
- `y` will hold the outcome column we want to predict, our **target**

`X` and `y` are the standard names for these data that you will see in machine learning.

### <font color=0D9488>**Question 6:**</font> Create `X` and `y` from `ces_sup_clean`. Assign the `Asthma` column as the target variable `y`, and use `possible_feature_cols` as `X`. Then print the shapes of `X` and `y`.

In [ ]:
# Your solution here!

A quick reminder about shapes:

- `y.shape` gives `(number_of_rows, )` because `y` is a single column
- `X.shape` gives `(number_of_rows, number_of_feature_columns)`

So `X` and `y` should have the same number of rows, because each row of features must line up with one row of target values.

Let's look at them to make sure we understand before proceeding.

In [ ]:
print("Target dataset y:")
display(y.head())

print("\nFeature dataset X:")
display(X.head())

## Step 10: Split the rows into training and test sets

Up to this point, we have been organizing the data by **column role**. Now we are going to do a different kind of split: this time we will split the **rows** into two groups.

- the **training set**, which the model will learn from later
- the **test set**, which we will hold aside until after training so we can check how well the model performs on rows it has not seen before

This is an important habit in supervised learning. If we trained and tested the model on the exact same rows, the results could look better than they really are. By saving some rows for testing, we get a more honest check of how well the model may generalize to new data.

Usually, the training set is larger than the test set. That is because the model needs enough data to learn patterns, but we also want to keep some rows aside for evaluation. A common starting choice is to use **80%** of the rows for training and **20%** for testing.

This split is usually done **randomly**, so the training and test sets are mixed samples from the same dataset rather than one block of rows from the top and one block from the bottom. In `sklearn`, we often use a fixed `random_state` so that we get the same split of rows each time we run the notebook. That makes our work reproducible.

### Predict before running

Before you run the next code cell, pause and think about these questions:

- If `X` and `y` start with the same number of rows, what should be true about `X_train` and `y_train`?
- Why do we split `X` and `y` together instead of splitting them separately?
- If some feature columns still have missing values, should we handle those missing values before or after the split?

In the code below, you will see a new function called `train_test_split()` from `sklearn`. You do not need to memorize it yet. For now, focus on what it does: it takes the rows in `X` and `y` and splits them into matching training and test sets.

In [ ]:
# Worked example
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

Visually, we can think about the two data-splitting steps in the supervised machine learning workflow like the picture below.

We begin with one dataset, separate the target `y` from the feature columns `X`, and then split the rows into training and test sets.

<img src="https://www.reneshbedre.com/assets/posts/others/split_train_test.webp" width="600" height="400">

If this still feels a little confusing, that is okay. We will practice this several times.

### <font color=0D9488>**Question 7:**</font> Just for practice, create a second train/test split using `test_size=0.25` and `random_state=42`. Name the outputs `X_train_25`, `X_test_25`, `y_train_25`, and `y_test_25`. Then print their shapes.

In [ ]:
# Your solution here!

### <font color=0D9488>**Question 8:**</font> Now in the original 80/20% split, check whether there are any missing values in `y_train`, `y_test`, `X_train`, and `X_test`. What is the main difference between the target data and the feature data at this point?

In [ ]:
# Your solution here!

**Your written solution here!**

## Step 11: Preview of the sklearn supervised workflow

Now that we have checked the remaining missing feature values, we are going to stop the hands-on workflow for today. We are not fitting a model yet.

Before we end, it is helpful to preview the structure you will see next time. In many sklearn supervised learning workflows, the pattern looks like this:

1. choose a target and a starting feature set
2. remove inappropriate columns and rows
3. build `X` and `y`
4. split into training and test sets
5. fit preprocessing steps on `X_train` only
6. apply those same preprocessing steps to `X_test`
7. fit a model using the training data
8. evaluate how well the model does on the test data
9. interpret the results carefully

That repeated structure will show up again and again in supervised learning. In the next notebook, we will continue this workflow by preparing the feature values, fitting a regression model, making predictions, and evaluating how well the model works.

### <font color=0D9488>**Question 9:**</font> Below, write a short reflection on how today's supervised workflow differs from the unsupervised workflow you used earlier in the course. What feels familiar, and what is new?

**Your solution here!**

# Congratulations, you have completed today's notebook!

## Key Takeaways:

- You transitioned from **unsupervised** to **supervised** machine learning.
- You learned that supervised learning can involve predicting either a **number** or a **category**.
- You explored the CalEnviroScreen dataset and used **Asthma** as a guided example of a tract-level target.
- You identified context columns, other outcome columns, and caution columns that should not be treated as ordinary features in this first workflow.
- You learned that **data leakage** happens when the model gets information that is too directly connected to the answer.
- You removed rows with missing target values.
- You built `X` and `y` in the format sklearn expects.
- You split the usable rows into training and test sets.

Today you completed the **setup stage** of a supervised machine learning workflow. You worked through how to understand the dataset, choose a target, remove inappropriate columns and rows, build `X` and `y`, and split the remaining data into training and test sets.

## Where This Fits in the ML Workflow

In **Notebook 4B**, you will continue this supervised learning workflow by choosing a smaller starting feature set, preprocessing the training data, and fitting your first supervised model: **linear regression**.

We spend time on setup because, in machine learning, the model can only learn from the data we give it. Choices about target selection, leakage, missing values, feature choice, and training/test splitting all affect whether the workflow is meaningful and whether the evaluation is fair.

Next time, you will use the supervised-learning setup from today to move from data preparation into real predictive modeling.